# Focus Group Discussion — GPU transcription and analysis

Set Colab runtime to **T4 GPU**, then run the cells in order. The ZIP download includes the mastered compact `audio.wav`, transcript, Q&A/themes report, CSV, JSON and DOCX. Silence >1.2 seconds is reduced to 0.35 seconds; timestamps refer to compact audio.

Audio sources have no visual frames: video analysis is explicitly marked unavailable. Review automatic transcript/analysis before publishing research findings.


In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install faster-whisper pandas python-docx transformers accelerate bitsandbytes sentencepiece
import json, subprocess, shutil, torch, pandas as pd
from pathlib import Path
from google.colab import files
assert torch.cuda.is_available(), 'Use Runtime > Change runtime type > T4 GPU'
WORK=Path('/content/fgd'); WORK.mkdir(exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
uploaded=files.upload(); INPUT=next(iter(uploaded))
assert Path(INPUT).suffix.lower() in {'.mp3','.wav','.m4a','.aac','.flac','.ogg','.opus','.mp4','.mov','.mkv'}
print('Uploaded:',INPUT)


In [ ]:
# Clear speech, compact long silences, normalize to -16 LUFS and protect peaks.
FILTER='highpass=f=70,lowpass=f=7600,afftdn=nr=10:nf=-35:tn=1,acompressor=threshold=-20dB:ratio=2.5:attack=20:release=250,silenceremove=stop_periods=-1:stop_duration=1.2:stop_threshold=-38dB:stop_silence=0.35,loudnorm=I=-16:TP=-1.5:LRA=11'
FINAL='/content/audio.wav'
cmd=['ffmpeg','-hide_banner','-y','-i',INPUT,'-vn','-af',FILTER,'-ac','1','-ar','16000','-c:a','pcm_s16le',FINAL]
subprocess.run(cmd,check=True)
probe=subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration,size:stream=sample_rate,channels','-of','json',FINAL],text=True)
quality={'source':INPUT,'output':'audio.wav','processing':'high-pass 70 Hz; low-pass 7.6 kHz; gentle denoise; compression; -16 LUFS target; -1.5 dBTP target; silences >1.2 s reduced to 0.35 s','probe':json.loads(probe),'timestamp_note':'Transcript times use compact audio.wav.'}
print(json.dumps(quality,indent=2))


In [ ]:
from faster_whisper import WhisperModel
wm=WhisperModel('large-v3',device='cuda',compute_type='float16')
segments,info=wm.transcribe(FINAL,beam_size=5,best_of=5,temperature=0,word_timestamps=True,vad_filter=False,condition_on_previous_text=True)
rows=[{'id':i,'start':round(s.start,2),'end':round(s.end,2),'duration':round(s.end-s.start,2),'speaker':'SPEAKER_UNKNOWN','text':s.text.strip(),'avg_logprob':s.avg_logprob} for i,s in enumerate(segments,1)]
df=pd.DataFrame(rows); assert not df.empty,'No speech detected — reduce the silence threshold.'
def ts(x): return f'{int(x//3600):02}:{int(x%3600//60):02}:{int(x%60):02}'
transcript='\n'.join(f'[{ts(r.start)}–{ts(r.end)}] {r.speaker}: {r.text}' for r in df.itertuples())
(WORK/'transcript_timestamped.txt').write_text(transcript,encoding='utf-8'); df.to_csv(WORK/'transcription.csv',index=False,encoding='utf-8-sig')
print('Language:',info.language,round(info.language_probability,3)); print(transcript[:3000])


In [ ]:
# Free, local-on-GPU qualitative analysis. Speaker labels remain UNKNOWN unless manually verified.
from transformers import AutoTokenizer,AutoModelForCausalLM
tok=AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct')
llm=AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-3B-Instruct',torch_dtype=torch.float16,device_map='auto',load_in_4bit=True)
def ask(q,n=1600):
 m=[{'role':'system','content':'You are a rigorous qualitative researcher. Use only supplied transcript evidence. Do not invent quotes, identities, demographics, speaker labels, or video observations.'},{'role':'user','content':q}]
 x=tok.apply_chat_template(m,add_generation_prompt=True,return_tensors='pt').to(llm.device); y=llm.generate(x,max_new_tokens=n,do_sample=False,pad_token_id=tok.eos_token_id)
 return tok.decode(y[0][x.shape[1]:],skip_special_tokens=True)
lines=transcript.splitlines(); chunks=[]; cur=[]; n=0
for line in lines:
 if n+len(line)>9000 and cur: chunks.append('\n'.join(cur)); cur=[]; n=0
 cur.append(line); n+=len(line)+1
if cur: chunks.append('\n'.join(cur))
parts=[ask('Analyse this FGD section with headings: Questions and Answers; Themes and Subthemes; Agreement and Disagreement; Attitudes; Evidence with timestamps; Audio/voice limitations.\n\n'+c) for c in chunks]
analysis=ask('Synthesize the following FGD section analyses into a report with Executive Summary, Question–Answer Matrix, core themes/subthemes, agreement/disagreement, attitudes, timestamped evidence, research limitations, and this exact limitation: No video analysis is possible because the supplied source is audio-only. Do not add facts.\n\n'+'\n\n---\n'.join(parts),2200)
(WORK/'analysis.md').write_text(analysis,encoding='utf-8'); print(analysis)


In [ ]:
from docx import Document
OUT=Path('/content/FGD_COMPLETE_OUTPUT'); OUT.mkdir(exist_ok=True)
for src,name in [(FINAL,'audio.wav'),(WORK/'transcript_timestamped.txt','transcript_timestamped.txt'),(WORK/'transcription.csv','transcription.csv'),(WORK/'analysis.md','FGD_analysis.md')]: shutil.copy(src,OUT/name)
(OUT/'audio_quality.json').write_text(json.dumps(quality,indent=2),encoding='utf-8')
package={'audio_file':'audio.wav','language':info.language,'language_probability':float(info.language_probability),'segments':df.to_dict(orient='records'),'analysis':analysis,'video_analysis':'Unavailable: supplied source is audio-only.'}
(OUT/'FGD_research_package.json').write_text(json.dumps(package,ensure_ascii=False,indent=2),encoding='utf-8')
doc=Document(); doc.add_heading('Focus Group Discussion: Analysis',0); doc.add_heading('Qualitative findings',1)
for p in analysis.splitlines():
 if p.startswith('#'): doc.add_heading(p.lstrip('# '),min(3,len(p)-len(p.lstrip('#'))))
 elif p.strip(): doc.add_paragraph(p)
doc.add_page_break(); doc.add_heading('Timestamped transcript',1)
for p in transcript.splitlines(): doc.add_paragraph(p)
doc.save(OUT/'FGD_report.docx')
zipfile=shutil.make_archive('/content/FGD_COMPLETE_PACKAGE','zip',OUT); print(zipfile); print(*sorted(x.name for x in OUT.iterdir()),sep='\n'); files.download(zipfile)
